In [16]:
import pandas as pd
import networkx as nx

files = {
    "MAOL": r"C:\Users\vachi\Downloads\maol_1.1_edge_list.csv",
    "FAFB": r"C:\Users\vachi\Downloads\fafb_783_edge_list.csv",
    "BANC": r"C:\Users\vachi\Downloads\banc_626_edge_list (1).csv"
}

graphs = {}

for name, path in files.items():
    df = pd.read_csv(path)
    df.columns = ["source", "target"]
    graphs[name] = nx.from_pandas_edgelist(df, "source", "target", create_using=nx.DiGraph())

print("loaded")

loaded


In [17]:
def signature(G, n):
    if n not in G:
        return None

    out_n = len(list(G.successors(n)))
    in_n = len(list(G.predecessors(n)))

    # second-order flow context (VERY IMPORTANT)
    out2 = 0
    for v in list(G.successors(n))[:10]:
        out2 += len(list(G.successors(v)))

    return (out_n, in_n, out2)

In [18]:
from collections import defaultdict

sig_maps = {}

for name, G in graphs.items():
    mp = defaultdict(list)

    for n in G.nodes():
        s = signature(G, n)
        if s:
            mp[s].append(n)

    sig_maps[name] = mp

print("signature maps built")

signature maps built


In [19]:
shared_sigs = set(sig_maps["MAOL"]) & set(sig_maps["FAFB"]) & set(sig_maps["BANC"])

print("shared structural roles:", len(shared_sigs))

shared structural roles: 90


In [20]:
MAOL_nodes, FAFB_nodes, BANC_nodes = [], [], []

for s in list(shared_sigs)[:2000]:   # allow scale
    MAOL_nodes += sig_maps["MAOL"][s]
    FAFB_nodes += sig_maps["FAFB"][s]
    BANC_nodes += sig_maps["BANC"][s]

print(len(MAOL_nodes), len(FAFB_nodes), len(BANC_nodes))

647 2052 4930


In [23]:
N = min(len(MAOL_nodes), len(FAFB_nodes), len(BANC_nodes))

df = pd.DataFrame({
    "MAOL": MAOL_nodes[:N],
    "FAFB": FAFB_nodes[:N],
    "BANC": BANC_nodes[:N],
})

df.to_csv("network.csv", index=False)

print("saved network.csv with rows:", N)
df.head()

saved network.csv with rows: 647


,MAOL,FAFB,BANC
0,37865,720575940609719250,720575941513210371
1,18540,720575940618594117,720575941440141909
2,45678,720575940625144061,720575941553640676
3,23372,720575940612836245,720575941429726857
4,17300,720575940614969651,720575941451317575
